# LP Step 4 — Gold Layer
### Executive KPIs + Customer Scorecard + Carrier Performance

**Output tables:**
- `gold_executive_kpis` — Single-row company-wide KPI summary
- `gold_customer_scorecard` — 15 customers ranked by composite score (0-100)
- `gold_carrier_performance` — 8 carriers graded A through D

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

SILVER_ORDERS    = "supply_chain_dev.lp_analytics.silver_orders"
SILVER_ACCOUNTS  = "supply_chain_dev.lp_analytics.silver_accounts"
SILVER_SHIPMENTS = "supply_chain_dev.lp_analytics.silver_shipments"
SILVER_UNIFIED   = "supply_chain_dev.lp_analytics.silver_unified_customer_view"

GOLD_EXEC_KPIS       = "supply_chain_dev.lp_analytics.gold_executive_kpis"
GOLD_CUSTOMER_SCORE  = "supply_chain_dev.lp_analytics.gold_customer_scorecard"
GOLD_CARRIER_PERF    = "supply_chain_dev.lp_analytics.gold_carrier_performance"

print("Gold layer targets:")
print(f"  {GOLD_EXEC_KPIS}")
print(f"  {GOLD_CUSTOMER_SCORE}")
print(f"  {GOLD_CARRIER_PERF}")

Gold layer targets:
  supply_chain_dev.lp_analytics.gold_executive_kpis
  supply_chain_dev.lp_analytics.gold_customer_scorecard
  supply_chain_dev.lp_analytics.gold_carrier_performance


## Step 4.1 — Gold Executive KPIs
Company-wide summary combining SAP revenue metrics and TMS delivery metrics.
Result: 1 row — $2.47M revenue, 21.98% margin, 76.3% on-time, 89 overdue invoices.

In [0]:
from datetime import date

orders_df    = spark.table(SILVER_ORDERS)
shipments_df = spark.table(SILVER_SHIPMENTS)

# Revenue metrics from SAP
revenue_kpis = orders_df.agg(
    count("sap_order_id").alias("total_orders"),
    round(sum("revenue_usd"), 2).alias("total_revenue_usd"),
    round(sum("margin_usd"), 2).alias("total_margin_usd"),
    round(avg("margin_pct"), 2).alias("avg_margin_pct"),
    round(sum("freight_cost_usd"), 2).alias("total_freight_cost_usd"),
    countDistinct("customer_id").alias("active_customers"),
    sum(when(col("payment_status") == "OVERDUE", 1).otherwise(0)).alias("overdue_invoices"),
    sum(when(col("payment_status") == "PAID", col("invoice_amount_usd")).otherwise(0)).alias("collected_revenue_usd"),
    sum(when(col("order_status") == "CLOSED", 1).otherwise(0)).alias("closed_orders")
)

# Delivery metrics from TMS
shipment_kpis = shipments_df.agg(
    count("shipment_id").alias("total_shipments"),
    round(avg(col("on_time_delivery").cast("int")) * 100, 1).alias("overall_on_time_pct"),
    round(avg("delay_days"), 1).alias("avg_delay_days"),
    sum(when(col("delay_category") == "Severe Delay", 1).otherwise(0)).alias("severe_delay_count"),
    countDistinct("carrier").alias("total_carriers"),
    sum(when(col("customs_cleared") == False, 1).otherwise(0)).alias("customs_issues")
)

revenue_row  = revenue_kpis.collect()[0]
shipment_row = shipment_kpis.collect()[0]

kpi_data = [{
    "report_date": str(date.today()),
    "total_orders": revenue_row["total_orders"],
    "total_revenue_usd": revenue_row["total_revenue_usd"],
    "total_margin_usd": revenue_row["total_margin_usd"],
    "avg_margin_pct": revenue_row["avg_margin_pct"],
    "total_freight_cost_usd": revenue_row["total_freight_cost_usd"],
    "active_customers": revenue_row["active_customers"],
    "overdue_invoices": revenue_row["overdue_invoices"],
    "collected_revenue_usd": revenue_row["collected_revenue_usd"],
    "total_shipments": shipment_row["total_shipments"],
    "overall_on_time_pct": shipment_row["overall_on_time_pct"],
    "avg_delay_days": shipment_row["avg_delay_days"],
    "severe_delay_count": shipment_row["severe_delay_count"],
    "total_carriers": shipment_row["total_carriers"],
    "customs_issues": shipment_row["customs_issues"]
}]

kpi_df = spark.createDataFrame(kpi_data)
kpi_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_EXEC_KPIS)

print("Executive KPIs saved!")
display(spark.table(GOLD_EXEC_KPIS).select(
    "report_date", "total_orders", "total_revenue_usd",
    "avg_margin_pct", "overall_on_time_pct", "overdue_invoices"
))

active_customers,avg_delay_days,avg_margin_pct,collected_revenue_usd,customs_issues,overall_on_time_pct,overdue_invoices,report_date,severe_delay_count,total_carriers,total_freight_cost_usd,total_margin_usd,total_orders,total_revenue_usd,total_shipments
15,2.0,21.98,617460.75,97,76.3,89,Column<'current_date()'>,55,8,1956590.22,508669.12,300,2465259.34,400


Executive KPIs saved!


## Step 4.2 — Gold Customer Scorecard
Composite score per customer: Revenue (40%) + Service/OTD (30%) + NPS (20%) + Contract (10%).
Tiers: Platinum (70+) | Gold (50-69) | Silver (30-49) | Bronze (<30)

In [0]:
# ── GOLD 2: CUSTOMER SCORECARD ─────────────────────────────────────────
# Ranks every customer by a composite score combining
# revenue contribution + service performance + relationship health

from pyspark.sql.window import Window

unified_df = spark.table(SILVER_UNIFIED)

# Calculate max values for normalization
max_revenue = unified_df.agg(max("total_revenue_usd")).collect()[0][0]
max_orders = unified_df.agg(max("total_orders")).collect()[0][0]

scorecard_df = unified_df \
    .withColumn("revenue_score",
        round((col("total_revenue_usd") / max_revenue) * 40, 2)) \
    .withColumn("service_score",
        round((col("on_time_pct") / 100) * 30, 2)) \
    .withColumn("relationship_score",
        round((col("nps_score") / 100) * 20, 2)) \
    .withColumn("contract_score",
        round((col("renewal_probability_pct") / 100) * 10, 2)) \
    .withColumn("composite_score",
        round(
            col("revenue_score") +
            col("service_score") +
            col("relationship_score") +
            col("contract_score"), 2
        )
    ) \
    .withColumn("customer_tier",
        when(col("composite_score") >= 70, "Platinum")
        .when(col("composite_score") >= 50, "Gold")
        .when(col("composite_score") >= 30, "Silver")
        .otherwise("Bronze")
    ) \
    .withColumn("rank",
        row_number().over(Window.orderBy(desc("composite_score")))
    ) \
    .withColumn("health_flag",
        when(
            (col("account_status") == "At Risk") |
            (col("nps_score") < 30) |
            (col("overdue_invoices") > 7),
            "NEEDS ATTENTION"
        ).otherwise("HEALTHY")
    ) \
    .select(
        "rank", "customer_id", "account_name", "segment",
        "account_status", "total_revenue_usd", "total_orders",
        "on_time_pct", "nps_score", "contract_value_usd",
        "renewal_probability_pct", "overdue_invoices",
        "revenue_score", "service_score", "relationship_score",
        "contract_score", "composite_score", "customer_tier",
        "health_flag", "assigned_rep"
    )

scorecard_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_CUSTOMER_SCORE)

print(f"Customer Scorecard: {spark.table(GOLD_CUSTOMER_SCORE).count()} customers")
display(spark.table(GOLD_CUSTOMER_SCORE).select(
    "rank", "account_name", "customer_tier", "composite_score",
    "total_revenue_usd", "on_time_pct", "nps_score",
    "overdue_invoices", "health_flag"
).orderBy("rank"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer Scorecard: 15 customers


rank,account_name,customer_tier,composite_score,total_revenue_usd,on_time_pct,nps_score,overdue_invoices,health_flag
1,Amazon Logistics,Platinum,79.77,265636.0,83.9,50,9,NEEDS ATTENTION
2,Best Buy Logistics,Platinum,73.36,182697.54,81.5,86,7,HEALTHY
3,Target Distribution,Platinum,70.72,203850.42,69.4,77,3,NEEDS ATTENTION
4,Zara Supply Chain,Gold,65.4,178119.08,84.6,40,4,HEALTHY
5,Unilever Logistics,Gold,64.9,170759.1,83.3,25,7,NEEDS ATTENTION
6,IKEA Distribution,Gold,64.16,114630.26,90.0,80,6,HEALTHY
7,Procter & Gamble,Gold,63.38,195103.26,80.0,21,7,NEEDS ATTENTION
8,Walmart Supply Chain,Gold,62.29,168664.08,64.3,72,6,HEALTHY
9,Home Depot Freight,Gold,60.94,194681.54,70.4,26,8,NEEDS ATTENTION
10,Tesla Parts Logistics,Gold,60.76,159033.65,66.7,45,5,HEALTHY


## Step 4.3 — Gold Carrier Performance
Carrier grading based on on-time delivery: A (>=85%) | B (>=75%) | C (>=65%) | D (<65%)
Result: 8 carriers ranked — DHL Global leads at 83.3%, Evergreen last at 57.8%.

In [0]:
# ── GOLD 3: CARRIER PERFORMANCE ────────────────────────────────────────
shipments_df = spark.table(SILVER_SHIPMENTS)

carrier_df = shipments_df \
    .groupBy("carrier") \
    .agg(
        count("shipment_id").alias("total_shipments"),
        round(avg(col("on_time_delivery").cast("int")) * 100, 1).alias("on_time_pct"),
        round(avg("delay_days"), 2).alias("avg_delay_days"),
        round(avg("transit_days_actual"), 1).alias("avg_transit_days"),
        round(avg("freight_charges_usd"), 2).alias("avg_freight_cost_usd"),
        round(sum("freight_charges_usd"), 2).alias("total_freight_spend_usd"),
        sum(when(col("delay_category") == "Severe Delay", 1).otherwise(0)).alias("severe_delays"),
        sum(when(col("delay_category") == "On Time", 1).otherwise(0)).alias("on_time_count"),
        countDistinct("customer_id").alias("customers_served"),
        countDistinct("trade_lane").alias("trade_lanes_covered"),
        sum(when(col("customs_cleared") == False, 1).otherwise(0)).alias("customs_failures")
    ) \
    .withColumn("carrier_grade",
        when(col("on_time_pct") >= 85, "A")
        .when(col("on_time_pct") >= 75, "B")
        .when(col("on_time_pct") >= 65, "C")
        .otherwise("D")
    ) \
    .withColumn("recommendation",
        when(col("carrier_grade") == "A", "Preferred Carrier")
        .when(col("carrier_grade") == "B", "Approved Carrier")
        .when(col("carrier_grade") == "C", "Monitor Closely")
        .otherwise("Review Contract")
    ) \
    .orderBy(desc("on_time_pct"))

carrier_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_CARRIER_PERF)

print(f"Carrier Performance: {spark.table(GOLD_CARRIER_PERF).count()} carriers")
display(spark.table(GOLD_CARRIER_PERF).select(
    "carrier", "total_shipments", "on_time_pct",
    "avg_delay_days", "avg_freight_cost_usd",
    "severe_delays", "carrier_grade", "recommendation"
).orderBy(desc("on_time_pct")))

Carrier Performance: 8 carriers


carrier,total_shipments,on_time_pct,avg_delay_days,avg_freight_cost_usd,severe_delays,carrier_grade,recommendation
DHL Global,48,83.3,1.63,13925.49,5,B,Approved Carrier
FedEx Freight,48,81.3,1.94,13265.87,8,B,Approved Carrier
Hapag-Lloyd,50,80.0,1.76,11922.35,6,B,Approved Carrier
MSC,47,78.7,1.7,11968.86,6,B,Approved Carrier
UPS Supply Chain,57,77.2,2.21,12303.83,9,B,Approved Carrier
COSCO,51,76.5,1.69,13646.53,5,B,Approved Carrier
CMA CGM,54,74.1,1.61,13041.52,5,C,Monitor Closely
Evergreen,45,57.8,3.62,14819.67,11,D,Review Contract
